In [1]:
import pandas as pd
import requests

In [2]:
emlap_metadata = pd.read_csv("/srv/data/tome/tome-corpus/emlap_metadata.csv", sep=";") # , index=True)

In [3]:
emlap_metadata.head()

,working_title,filenames,no.,is_done,is_noscemus,if_noscemus_id,AUTHORSHIP,is_one_author,#if more than 1 author skip section and choose compendium below,is_author_known,...,publisher_comments,CONTENTS,genre,subject,SOURCE OF FILE,link,source_of_file,origin_of_copy,other_notes,tokens_N
0,"Augurello, Chrysopoeia",100001_Augurello1515_Chrysopoeia_GB_Noscemus,100001,True,True,713324.0,NaN,True,NaN,True,...,NaN,NaN,didactic poem,alchemy,NaN,https://wiki.uibk.ac.at/noscemus/Chrysopoeia,GB,Noscemus,NaN,23718
1,"Pseudo-Lull, Secretis",100002_Pseudo-Lull1518_De secretis_naturae_MDZ...,100002,True,False,NaN,NaN,True,NaN,True,...,NaN,NaN,treatise,"alchemy, medicine",NaN,https://www.digitale-sammlungen.de/en/view/bsb...,MDZ,MBS,NaN,24673
2,"Pantheus, Ars Transmutatione",100003_Pantheus1518_Ars_Transmutationis_Metall...,100003,True,False,NaN,NaN,True,NaN,True,...,NaN,NaN,treatise,alchemy,NaN,https://www.google.co.uk/books/edition/Ars_Tra...,GB,BL,NaN,8646
3,"Anon, Vera alchemiae",100004_Anon1561_Verae_Alchemiae_MDZ_MBS,100004,True,False,NaN,NaN,True,NaN,True,...,NaN,NaN,"compendium, florilegium",alchemy,NaN,https://mdz-nbn-resolving.de/details:bsb10141168,MDZ,MBS,NaN,3521
4,"Pantheus, Voarchadumia",100005_Pantheus1530_Voarchadumia_ONB,100005,True,False,NaN,NaN,True,NaN,True,...,NaN,NaN,treatise,alchemy,NaN,https://data.onb.ac.at/rep/10588E49,ONB,ONB,NaN,20386


In [4]:
def wd_by_an_identifier(property, idstring):
    """
    Query Wikidata for an entity with the given TLG author ID (property P1266).
    Returns JSON results from the WDQS endpoint.
    """
    query = f"""
    SELECT ?item ?itemLabel WHERE {{
      ?item wdt:{property} "{idstring}" .
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """

    url = "https://query.wikidata.org/sparql"
    headers = {"Accept": "application/sparql-results+json"}

    response = requests.get(url, params={"query": query, "format": "json"}, headers=headers)
    response.raise_for_status()

    return response.json()

def get_wd_authors(viaf):
    author_wd = ""
    try:
        author_wd =  wd_by_an_identifier("P214", viaf)["results"]["bindings"][0]["item"]["value"].rpartition("/")[2]
    except:
        pass
    return author_wd

In [5]:
emlap_metadata["aurhor_wd"] = emlap_metadata["author_viaf"].apply(get_wd_authors)

In [6]:
emlap_metadata.to_csv("../data/emlap_metadata.csv", sep=";") # , index=True)

In [7]:
emlap_metadata.to_csv("/srv/data/tome/tome-corpus/emlap_metadata.csv", sep=";") # , index=True)